# ADIM 6 — Koşullu Transfer Probe (barajlı, dürüst) + semantik duyarlılık

Bir sette eğitilen modelin BAŞKA sette ne kadar genellediğini ölçer. İki tip:
**(a) sektör-içi** (Cell2Cell↔Iranian, iki yön) ve **(b) sektör-ötesi** (telco↔bank,
iki yön). Setler birleştirilmez; transfer = "kaynakta fit, hedefte predict". Scaler
yalnız kaynakta fit (sızıntı yok). Model: HAM LightGBM. Önceden tanımlı baraj
(DAHİL/KISMÎ/ZAYIF) — sonuca göre eğilmez.

**Ek (duyarlılık):** önem-temelli temsilci seçimi anlamca farklı kolonları eşleyebilir
(kullanım=MonthlyMinutes vs Status). Bu yüzden telekom-içi çift için temsilcileri
**semantik olarak elle sabitleyip** transferi tekrar koşuyoruz ve iki varyantı
kıyaslıyoruz. Ağır mantık `src/transfer.py`'de.

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd


def _bul_kok():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "config.yaml").exists():
            return c
    raise RuntimeError("config.yaml bulunamadı")


KOK = _bul_kok()
if str(KOK) not in sys.path:
    sys.path.insert(0, str(KOK))

warnings.filterwarnings("ignore")
from src import config as cfg
from src import plotstyle as ps
from src import strings_tr as S
from src import transfer as tr

np.random.seed(cfg.SEED)
ps.uygula()
cfg.klasorleri_hazirla()
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)

CIKTI = []


def yaz(s=""):
    print(s)
    CIKTI.append(str(s))

## 1. Veri yükleme

In [2]:
veriler = {k: pd.read_csv(cfg.PROCESSED / f"{k}_clean.csv") for k in cfg.DATASETS}
for k, d in veriler.items():
    yaz(f"{k:11s} {d.shape}")

telco       (7043, 20)
cell2cell   (51047, 57)
ecommerce   (3941, 11)
iranian     (3150, 14)
bank        (10000, 11)


## 2. Önem-temelli transfer (4 senaryo)

In [3]:
yaz(S.MSG["bolum"].format(ad="TRANSFER PROBE (önem-temelli)"))
onem = []
for anahtar, ad, ks, ht, tip in tr.SENARYOLAR:
    r = tr.calistir_senaryo(anahtar, ad, ks, ht, tip, veriler, cfg.SEED)
    onem.append(r)
    yaz(S.MSG6["senaryo"].format(ad=r["ad"], yon=r["tip"], k=len(r["ortak"]),
        tr=r["transfer"], ref=r["ref"], tv=r["trivial"], oran=r["oran"],
        karar=S.TRANSFER_KARAR[r["karar"]]))

===== TRANSFER PROBE (önem-temelli) =====


Cell2Cell -> Iranian (sektör-içi): ortak kavram=5 | transfer PR-AUC=0.153 | ref=0.845 | trivial=0.157 | oran=0.18 -> ZAYIF


Iranian -> Cell2Cell (sektör-içi): ortak kavram=5 | transfer PR-AUC=0.278 | ref=0.404 | trivial=0.288 | oran=0.69 -> ZAYIF


telco -> bank (sektör-ötesi): ortak kavram=3 | transfer PR-AUC=0.237 | ref=0.450 | trivial=0.204 | oran=0.53 -> KISMÎ


bank -> telco (sektör-ötesi): ortak kavram=3 | transfer PR-AUC=0.207 | ref=0.622 | trivial=0.265 | oran=0.33 -> ZAYIF


## 3. Semantik-eşlemeli transfer (telekom-içi A1/A2)
Temsilciler önem'e göre DEĞİL, aynı operasyonel olguya göre elle sabit:
MonthlyMinutes↔Seconds of Use, PeakCallsInOut↔Frequency of use,
CustomerCareCalls↔Complains, MonthsInService↔Subscription Length, MonthlyRevenue↔Customer Value.

In [4]:
yaz(S.MSG["bolum"].format(ad="TRANSFER PROBE (semantik eşleme)"))
semantik = []
for anahtar, ad, ks, ht, tip in tr.SENARYOLAR:
    if tip != "sektör-içi":
        continue
    r = tr.calistir_semantik(anahtar, ad, ks, ht, veriler, cfg.SEED)
    semantik.append(r)
    yaz(S.MSG6["semantik"].format(ad=r["ad"], k=len(r["ortak"]), tr=r["transfer"],
        ref=r["ref"], tv=r["trivial"], oran=r["oran"], karar=S.TRANSFER_KARAR[r["karar"]]))
    if r["atlanan"]:
        yaz(S.MSG6["atlanan"].format(liste=r["atlanan"]))

===== TRANSFER PROBE (semantik eşleme) =====


Cell2Cell -> Iranian [semantik]: kavram=5 | transfer PR-AUC=0.281 | ref=0.870 | trivial=0.157 | oran=0.32 -> KISMÎ


Iranian -> Cell2Cell [semantik]: kavram=5 | transfer PR-AUC=0.270 | ref=0.394 | trivial=0.288 | oran=0.68 -> ZAYIF


## 4. Tablolar
`transfer_results.csv` (önem + semantik, esleme_tipi kolonuyla) ve eşleme haritaları.

In [5]:
sonuc_df = tr.tablo_sonuc(onem + semantik)
tr.tablo_feature_map(onem)
tr.tablo_feature_map_semantic()
yaz(S.MSG["bolum"].format(ad="TRANSFER SONUÇLARI (önem + semantik)"))
yaz(sonuc_df.to_string(index=False))
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "transfer_results.csv"))
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "transfer_feature_map_semantic.csv"))

===== TRANSFER SONUÇLARI (önem + semantik) =====
Senaryo                  Yön  Eşleme tipi  Ortak kavram sayısı  Transfer PR-AUC  In-domain ref PR-AUC  Tam-feature ref PR-AUC  Trivial PR-AUC  Koruma oranı  Duyarlılık  Kesinlik     F1 Karar
     A1 Cell2Cell -> Iranian önem-temelli                    5           0.1531                0.8449                  0.9576          0.1571         0.181      0.4242    0.1308 0.2000 ZAYIF
     A2 Iranian -> Cell2Cell önem-temelli                    5           0.2783                0.4035                  0.4661          0.2882         0.690      0.8214    0.2829 0.4209 ZAYIF
     B1        telco -> bank önem-temelli                    3           0.2373                0.4503                  0.7065          0.2037         0.527      0.7545    0.2408 0.3651 KISMÎ
     B2        bank -> telco önem-temelli                    3           0.2075                0.6219                  0.6635          0.2654         0.334      0.0000    0.0000 0.0000 ZA

## 5. Figürler

In [6]:
yaz(S.MSG["bolum"].format(ad="FİGÜRLER"))
y1 = tr.figur_prauc(onem)
y2 = tr.figur_retention(onem)
y3 = tr.figur_semantic_vs_importance(onem, semantik)
for y in (y1, y2, y3):
    yaz(S.MSG["kayit"].format(yol=y))

===== FİGÜRLER =====


Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_transfer/transfer_prauc_comparison.png
Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_transfer/transfer_retention_ratio.png
Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_transfer/transfer_semantic_vs_importance.png


## 6. Özet — semantik eşleme transferi kurtardı mı? (karar kullanıcıda)

In [7]:
yaz(S.MSG["bolum"].format(ad="ÖZET — semantik vs önem-temelli"))
onem_map = {r["anahtar"]: r for r in onem}
kurtaran = []
for r in semantik:
    o = onem_map[r["anahtar"]]
    yaz(f"  {r['anahtar']} {r['ad']:22s}: önem oran={o['oran']:.2f} ({S.TRANSFER_KARAR[o['karar']]}) "
        f"-> semantik oran={r['oran']:.2f} ({S.TRANSFER_KARAR[r['karar']]})  "
        f"[transfer PR-AUC {o['transfer']:.3f} -> {r['transfer']:.3f}]")
    if r["karar"] == "dahil" or r["oran"] >= o["oran"] + 0.15:
        kurtaran.append(r["anahtar"])

dahil_var = any(r["karar"] == "dahil" for r in semantik)
yaz("")
if not dahil_var:
    yaz("SONUÇ: Semantik eşleme ile DE hiçbir senaryo baraja ulaşmıyor (DAHİL yok). "
        "Bulgu sağlamlaştı: zayıf transfer eşleştirme artefaktı değil — sürücüler "
        "gerçekten setler/sektörler arası taşınmıyor (RQ2'yi güçlendirir).")
else:
    yaz(f"SONUÇ: Semantik eşleme transferi toparladı ({kurtaran}); önceki zayıflık "
        "kısmen önem-temelli eşleme artefaktıymış — yeni nüans.")
yaz("")
yaz(S.MSG6["bitti"])

_log = cfg.LOGS / "adim6_ozet.log"
mevcut = _log.read_text(encoding="utf-8") if _log.exists() else ""
_log.write_text(mevcut + "\n\n# ==== SEMANTİK EK ====\n" + "\n".join(CIKTI) + "\n", encoding="utf-8")
print(S.MSG["kayit"].format(yol=_log))

===== ÖZET — semantik vs önem-temelli =====
  A1 Cell2Cell -> Iranian  : önem oran=0.18 (ZAYIF) -> semantik oran=0.32 (KISMÎ)  [transfer PR-AUC 0.153 -> 0.281]
  A2 Iranian -> Cell2Cell  : önem oran=0.69 (ZAYIF) -> semantik oran=0.68 (ZAYIF)  [transfer PR-AUC 0.278 -> 0.270]

SONUÇ: Semantik eşleme ile DE hiçbir senaryo baraja ulaşmıyor (DAHİL yok). Bulgu sağlamlaştı: zayıf transfer eşleştirme artefaktı değil — sürücüler gerçekten setler/sektörler arası taşınmıyor (RQ2'yi güçlendirir).

ADIM 6 (transfer) tamamlandı. Yorum/karar kullanıcıya bırakıldı. Sağlamlık/yazım (Adım 7) yapılmadı.
Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/logs/adim6_ozet.log
